# 🏆 Mini Project: Scouting the Academy's Next Star (NumPy)

### The Scenario

You are a **data analyst for a sports academy**. The head coach is putting together a **6-player squad** for an upcoming regional tournament, and has given you a fixed set of rules to follow so every analyst on the team produces a consistent, comparable report.

You have performance data for 12 players across 4 physical metrics. Follow the steps below exactly — the weights and selection rule are fixed by the coaching staff, so your final squad should match a single correct answer. Your own input comes in at the very end, in how you *explain* the result.

**Deliverable:** a ranked, structured record of players, the final 6-player squad (following the fixed rule below), and a short written justification.


## 1. The Data

Run the cell below — this is your starting dataset. Don't change it directly; if you need to adjust anything, work on a copy.

| Column | Meaning |
|---|---|
| 0 | Sprint Speed (km/h) |
| 1 | Endurance Score (0–100) |
| 2 | Strength Score (0–100) |
| 3 | Agility Score (0–100) |


In [1]:
import numpy as np

players = np.array([
    "Adam", "Mostafa", "Yara", "Hana", "Ziad", "Malak",
    "Omar", "Farida", "Karim", "Nada", "Tarek", "Reem"
])

positions = np.array([
    "Forward", "Defender", "Midfielder", "Forward", "Goalkeeper", "Defender",
    "Midfielder", "Forward", "Defender", "Midfielder", "Forward", "Goalkeeper"
])

# 12 players x 4 metrics: [Sprint Speed, Endurance, Strength, Agility]
performance = np.array([
    [28.5, 82, 75, 88],
    [26.0, 90, 88, 70],
    [30.2, 78, 60, 92],
    [27.8, 85, 70, 80],
    [24.5, 95, 92, 65],
    [25.9, 70, 65, 75],
    [29.1, 80, 55, 90],
    [31.0, 60, 50, 95],
    [23.8, 88, 90, 60],
    [28.9, 83, 68, 85],
    [27.2, 76, 72, 78],
    [22.9, 93, 95, 58],
])

print(players.shape, positions.shape, performance.shape)

(12,) (12,) (12, 4)


## 2. Explore & Clean

Before ranking anyone, get familiar with the data — and fix a known issue.

1. Look at the shape and structure of `performance`. Pick a couple of players and compare their rows by eye using indexing.
2. There is a known data-entry glitch: **any Strength score above 90 should be capped at exactly 90.** Fix this on a **copy** of `performance` (call it `performance_clean`), so you don't lose the original data.

   💡 `np.clip(array, min, max)` caps values into a range — pass `None` for a bound you don't want to limit.


In [3]:
# Explore the data, then create performance_clean here
# --- Explore ---
print("Shape:", performance.shape)
print("Adam's row:", performance[0])
print("Ziad's row:", performance[4])

# --- Clean: cap Strength (column index 2) above 90 to exactly 90 ---
performance_clean = performance.copy()
performance_clean[:, 2] = np.clip(performance_clean[:, 2], None, 90)

print(performance_clean)


Shape: (12, 4)
Adam's row: [28.5 82.  75.  88. ]
Ziad's row: [24.5 95.  92.  65. ]
[[28.5 82.  75.  88. ]
 [26.  90.  88.  70. ]
 [30.2 78.  60.  92. ]
 [27.8 85.  70.  80. ]
 [24.5 95.  90.  65. ]
 [25.9 70.  65.  75. ]
 [29.1 80.  55.  90. ]
 [31.  60.  50.  95. ]
 [23.8 88.  90.  60. ]
 [28.9 83.  68.  85. ]
 [27.2 76.  72.  78. ]
 [22.9 93.  90.  58. ]]


## 3. Build the Talent Score

The coaching staff has already agreed on how much each metric should count toward a player's overall Talent Score, so every scout's numbers line up. This tournament is a fast-paced format, so Sprint Speed and Agility are weighted highest:

| Metric | Weight |
|---|---|
| Sprint Speed | ×1.5 |
| Endurance | ×0.7 |
| Strength | ×0.5 |
| Agility | ×1.3 |

Combine `performance_clean` with this weight array using **broadcasting**, then sum across the metrics (per player) to get one `talent_score` value per player.


In [4]:
weights = np.array([1.5, 0.7, 0.5, 1.3])  # Sprint Speed, Endurance, Strength, Agility

talent_score = (performance_clean * weights).sum(axis=1)
print(talent_score)

[252.05 237.   249.5  240.2  232.75 217.85 244.15 237.   220.3  245.95
 231.4  219.85]


## 4. Package Everything into a Structured Array

Right now you have three separate arrays (`players`, `positions`, `talent_score`) that all describe the same 12 people — that's exactly the situation structured arrays are built for.

1. Build a structured array called `Squad` combining all three, with fields: `name` (string), `position` (string), and `score` (float).

   💡 One way: build it from a list of tuples using `dtype=[("name","U10"), ("position","U12"), ("score","f4")]`. You can construct the list of tuples with a loop over the index, e.g. `[(players[i], positions[i], talent_score[i]) for i in range(len(players))]`.

2. Explore it: print the field names, the shape, and confirm you can access `Squad["name"]` and `Squad["score"]` correctly.


In [5]:
squad_dtype = [("name", "U10"), ("position", "U12"), ("score", "f4")]

squad_data = [(players[i], positions[i], talent_score[i]) for i in range(len(players))]

Squad = np.array(squad_data, dtype=squad_dtype)

print("Field names:", Squad.dtype.names)
print("Shape:", Squad.shape)
print("Names:", Squad["name"])
print("Scores:", Squad["score"])

Field names: ('name', 'position', 'score')
Shape: (12,)
Names: ['Adam' 'Mostafa' 'Yara' 'Hana' 'Ziad' 'Malak' 'Omar' 'Farida' 'Karim'
 'Nada' 'Tarek' 'Reem']
Scores: [252.05 237.   249.5  240.2  232.75 217.85 244.15 237.   220.3  245.95
 231.4  219.85]


## 5. Filter & Sort by Field

Use field-based access to answer these:

1. Which players are Goalkeepers? (filter `Squad` where `position == "Goalkeeper"`)
2. Sort the whole `Squad` array by `score`, from highest to lowest, and print the result. (`np.sort` sorts ascending by default — think about how to reverse it.)
3. Convert `Squad` to a **record array** and use dot notation (`.name`, `.score`) to print the top 3 names and scores from your sorted result.


In [6]:
# Filter, sort, and use a record array here
# 1. Goalkeepers only
goalkeepers = Squad[Squad["position"] == "Goalkeeper"]
print("Goalkeepers:\n", goalkeepers)

# 2. Sort by score, highest to lowest
squad_sorted = np.sort(Squad, order="score")[::-1]
print("\nSorted squad:\n", squad_sorted)

# 3. Record array — dot notation for top 3
squad_rec = squad_sorted.view(np.recarray)
print("\nTop 3:")
for i in range(3):
    print(squad_rec.name[i], "-", squad_rec.score[i])


Goalkeepers:
 [('Ziad', 'Goalkeeper', 232.75) ('Reem', 'Goalkeeper', 219.85)]

Sorted squad:
 [('Adam', 'Forward', 252.05) ('Yara', 'Midfielder', 249.5 )
 ('Nada', 'Midfielder', 245.95) ('Omar', 'Midfielder', 244.15)
 ('Hana', 'Forward', 240.2 ) ('Mostafa', 'Defender', 237.  )
 ('Farida', 'Forward', 237.  ) ('Ziad', 'Goalkeeper', 232.75)
 ('Tarek', 'Forward', 231.4 ) ('Karim', 'Defender', 220.3 )
 ('Reem', 'Goalkeeper', 219.85) ('Malak', 'Defender', 217.85)]

Top 3:
Adam - 252.05
Yara - 249.5
Nada - 245.95


## 6. Select the Squad

Apply this fixed selection rule, exactly as written, so every scout arrives at the same squad:

1. Take the **6 players with the highest `score`**.
2. **Check the rule:** the squad must include **at least 1 Goalkeeper**.
3. **If it doesn't:** remove the *lowest-scoring* player currently in the top 6, and replace them with the *highest-scoring* Goalkeeper (even if that Goalkeeper's score is outside the top 6). **If it already does, leave the squad as is** — don't swap anyone.
4. Print your final 6-player squad (name, position, score).
5. Sanity check: print the average `score` of your final squad vs. the average `score` of the players left out.


In [7]:
# Select your final squad and sanity-check it here
# 1. Top 6 by score
final_squad = squad_sorted[:6].copy()
leftover = squad_sorted[6:].copy()

# 2. Check the rule: at least 1 Goalkeeper
has_gk = np.any(final_squad["position"] == "Goalkeeper")

if not has_gk:
    # 3. Swap lowest-scoring player in top 6 with the highest-scoring GK overall
    lowest_idx = np.argmin(final_squad["score"])
    best_gk = goalkeepers[np.argmax(goalkeepers["score"])]

    removed_player = final_squad[lowest_idx].copy()
    final_squad[lowest_idx] = best_gk

    # keep leftover list consistent: remove the GK we just added, add back the player we removed
    leftover = leftover[leftover["name"] != best_gk["name"]]
    leftover = np.append(leftover, removed_player)

    print(f"Swap made: {removed_player['name']} OUT, {best_gk['name']} IN")
else:
    print("Rule already satisfied — no swap needed.")

# 4. Print final squad
print("\nFinal Squad:")
for p in final_squad:
    print(f"{p['name']:10s} {p['position']:12s} {p['score']:.2f}")

# 5. Sanity check
print("\nAverage score - Final squad:", final_squad["score"].mean())
print("Average score - Left out:", leftover["score"].mean())


Swap made: Mostafa OUT, Ziad IN

Final Squad:
Adam       Forward      252.05
Yara       Midfielder   249.50
Nada       Midfielder   245.95
Omar       Midfielder   244.15
Hana       Forward      240.20
Ziad       Goalkeeper   232.75

Average score - Final squad: 244.09999
Average score - Left out: 227.23334


## 7. Final Report

In 4–6 sentences, write a short summary as if you were sending it to the coach. Include:
- The final squad (names)
- One sentence on why Endurance was weighted highest (in your own words, based on what a tournament format demands)
- Which player was swapped in/out to satisfy the Goalkeeper rule (if applicable), and what that costs the squad in terms of average score

*(Write this as a markdown cell or as comments — either is fine.)*


In [ ]:
# Your final report here
#
# The final squad selected for the tournament is: Adam, Yara, Nada, Omar, Hana, and Ziad.
#
# Sprint Speed and Agility were weighted highest because this is a fast-paced
# tournament format — players need to cover ground quickly, react fast, and
# change direction under pressure, so explosive speed and agility matter more
# in short, high-intensity matches than raw endurance.
#
# The top 6 by score had no Goalkeeper, so the rule kicked in: Mostafa (237.00,
# tied with Farida for the lowest score in the top 6) was removed and replaced
# with Ziad (232.75), the highest-scoring Goalkeeper. This swap lowered the
# squad's average score from 244.14 (average of the original top 6) to 244.10
# (average of the final squad) — a very small cost of about 0.68 points,
# which is a reasonable tradeoff to guarantee the team has a keeper on the pitch.